# 02 - Exploratory Data Analysis

**AurumMind AI - Phase 2: Market Analysis**

Comprehensive exploratory analysis of XAUUSD market data.

## Objectives
1. Analyze price distributions and patterns
2. Calculate and visualize returns
3. Study volatility behavior
4. Identify intraday patterns
5. Examine volume characteristics
6. Detect outliers and anomalies

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'src'))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from aurummind_ai.market_analysis import MarketStatistics, MarketVisualizer

# Configure display
pd.set_option('display.float_format', '{:.4f}'.format)

print("✓ Imports successful")

## 1. Load Data

In [ ]:
# Load 1H sample data
data_path = Path.cwd().parent / 'data' / 'raw' / 'samples' / 'XAUUSD_1h_sample.csv'
df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])

print(f"Loaded {len(df)} records")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
df.head()

## 2. Price Analysis

In [ ]:
# Candlestick chart
fig = MarketVisualizer.plot_candlestick(
    df,
    title="XAUUSD 1H - Full Dataset",
    show_volume=True
)
fig.show()

In [ ]:
# Price statistics
price_stats = MarketStatistics.calculate_price_statistics(df)

print("Price Statistics Summary:")
print("=" * 60)
for price_type, stats in price_stats.items():
    print(f"\n{price_type.upper()}:")
    print(f"  Mean:     ${stats['mean']:.2f}")
    print(f"  Median:   ${stats['median']:.2f}")
    print(f"  Std Dev:  ${stats['std']:.2f}")
    print(f"  Range:    ${stats['min']:.2f} - ${stats['max']:.2f}")
    print(f"  Skewness: {stats['skew']:.4f}")
    print(f"  Kurtosis: {stats['kurtosis']:.4f}")

In [ ]:
# Price distribution
fig = MarketVisualizer.plot_price_distribution(
    df,
    price_col="close",
    title="Close Price Distribution"
)
fig.show()

In [ ]:
# OHLC comparison
fig = MarketVisualizer.plot_ohlc_comparison(df)
fig.show()

## 3. Returns Analysis

In [ ]:
# Calculate returns
df['returns'] = MarketStatistics.calculate_returns(df, price_col='close', method='simple')
df['log_returns'] = MarketStatistics.calculate_returns(df, price_col='close', method='log')

# Return statistics
return_stats = MarketStatistics.calculate_return_statistics(df)

print("Return Statistics:")
print("=" * 60)
print(f"Mean Return:              {return_stats['mean']*100:.4f}%")
print(f"Median Return:            {return_stats['median']*100:.4f}%")
print(f"Std Dev:                  {return_stats['std']*100:.4f}%")
print(f"Min Return:               {return_stats['min']*100:.4f}%")
print(f"Max Return:               {return_stats['max']*100:.4f}%")
print(f"Skewness:                 {return_stats['skew']:.4f}")
print(f"Kurtosis:                 {return_stats['kurtosis']:.4f}")
print(f"\nPositive Returns:         {return_stats['positive_returns_pct']:.2f}%")
print(f"Negative Returns:         {return_stats['negative_returns_pct']:.2f}%")

In [ ]:
# Returns distribution
fig = MarketVisualizer.plot_returns_distribution(df, title="Simple Returns Distribution")
fig.show()

In [ ]:
# Cumulative returns
df['cumulative_returns'] = (1 + df['returns']).cumprod() - 1

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df['timestamp'],
        y=df['cumulative_returns'] * 100,
        mode='lines',
        name='Cumulative Returns',
        line=dict(color='steelblue', width=2)
    )
)

fig.update_layout(
    title="Cumulative Returns Over Time",
    xaxis_title="Date",
    yaxis_title="Cumulative Returns (%)",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

## 4. Volatility Analysis

In [ ]:
# Calculate volatility
df['volatility_20'] = MarketStatistics.calculate_volatility(df, window=20)
df['volatility_50'] = MarketStatistics.calculate_volatility(df, window=50)

# Plot volatility
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df['timestamp'],
        y=df['volatility_20'] * 100,
        mode='lines',
        name='20-period Volatility',
        line=dict(color='steelblue', width=2)
    )
)

fig.add_trace(
    go.Scatter(
        x=df['timestamp'],
        y=df['volatility_50'] * 100,
        mode='lines',
        name='50-period Volatility',
        line=dict(color='orange', width=2)
    )
)

fig.update_layout(
    title="Rolling Volatility Analysis",
    xaxis_title="Date",
    yaxis_title="Volatility (%)",
    template="plotly_white",
    hovermode="x unified"
)

fig.show()

In [ ]:
# Volatility statistics
print("Volatility Statistics (20-period):")
print("=" * 60)
vol_20 = df['volatility_20'].dropna()
print(f"Mean:    {vol_20.mean()*100:.4f}%")
print(f"Median:  {vol_20.median()*100:.4f}%")
print(f"Std Dev: {vol_20.std()*100:.4f}%")
print(f"Min:     {vol_20.min()*100:.4f}%")
print(f"Max:     {vol_20.max()*100:.4f}%")

## 5. Candle Analysis

In [ ]:
# Calculate candle features
df_candles = MarketStatistics.calculate_candle_features(df)

print("Candle Statistics:")
print("=" * 60)
print(f"Average Body Size:        ${df_candles['body_abs'].mean():.2f}")
print(f"Average Upper Wick:       ${df_candles['upper_wick'].mean():.2f}")
print(f"Average Lower Wick:       ${df_candles['lower_wick'].mean():.2f}")
print(f"Average Range:            ${df_candles['candle_range'].mean():.2f}")
print(f"Average Body Ratio:       {df_candles['body_ratio'].mean():.2%}")
print(f"\nBullish Candles:          {df_candles['is_bullish'].sum()} ({df_candles['is_bullish'].mean()*100:.2f}%)")
print(f"Bearish Candles:          {(1-df_candles['is_bullish']).sum()} ({(1-df_candles['is_bullish'].mean())*100:.2f}%)")

In [ ]:
# Candle range distribution
fig = go.Figure()
fig.add_trace(
    go.Histogram(
        x=df_candles['candle_range'],
        nbinsx=50,
        marker_color='steelblue'
    )
)

fig.update_layout(
    title="Candle Range Distribution",
    xaxis_title="Range (USD)",
    yaxis_title="Frequency",
    template="plotly_white",
    showlegend=False
)

fig.show()

## 6. Volume Analysis

In [ ]:
# Volume analysis
fig = MarketVisualizer.plot_volume_analysis(df, title="Volume Analysis")
fig.show()

In [ ]:
# Volume statistics
print("Volume Statistics:")
print("=" * 60)
print(f"Mean:             {df['volume'].mean():.2f}")
print(f"Median:           {df['volume'].median():.2f}")
print(f"Std Dev:          {df['volume'].std():.2f}")
print(f"Min:              {df['volume'].min():.2f}")
print(f"Max:              {df['volume'].max():.2f}")
print(f"\nZero Volume:      {(df['volume']==0).sum()} ({(df['volume']==0).mean()*100:.2f}%)")

## 7. Intraday Patterns

In [ ]:
# Intraday patterns
fig = MarketVisualizer.plot_intraday_patterns(df, title="Hourly Return Patterns")
fig.show()

In [ ]:
# Hour-based analysis
df['hour'] = pd.to_datetime(df['timestamp']).dt.hour

hourly_stats = df.groupby('hour').agg({
    'returns': ['mean', 'std', 'count'],
    'volume': 'mean',
    'candle_range': 'mean'
}).round(6)

print("\nHourly Statistics Summary:")
hourly_stats.head(10)

## 8. Outlier Detection

In [ ]:
# Detect outliers in returns
return_outliers = MarketStatistics.detect_outliers(
    df['returns'].dropna(),
    method='iqr',
    threshold=3.0
)

print(f"Return Outliers Detected: {return_outliers.sum()}")

if return_outliers.sum() > 0:
    outlier_returns = df.loc[return_outliers[return_outliers].index, ['timestamp', 'returns', 'close']]
    print("\nTop Outlier Returns:")
    print(outlier_returns.nlargest(5, 'returns'))

## 9. Comprehensive Summary Report

In [ ]:
# Generate full summary
summary = MarketStatistics.calculate_summary_report(df)

print("="*70)
print("COMPREHENSIVE MARKET ANALYSIS SUMMARY")
print("="*70)
print(f"\nDataset: {summary['general']['total_records']} records")
print(f"Period: {summary['general']['date_range']['start']} to {summary['general']['date_range']['end']}")
print(f"Duration: {summary['general']['date_range']['duration_days']} days")

print("\n" + "-"*70)
print("PRICE SUMMARY")
print("-"*70)
for price_type in ['close']:
    stats = summary['price_statistics'][price_type]
    print(f"\n{price_type.upper()}:")
    print(f"  Range: ${stats['min']:.2f} - ${stats['max']:.2f}")
    print(f"  Mean: ${stats['mean']:.2f}")
    print(f"  Volatility (Std): ${stats['std']:.2f}")

print("\n" + "-"*70)
print("RETURN SUMMARY")
print("-"*70)
ret_stats = summary['return_statistics']
print(f"Mean Return: {ret_stats['mean']*100:.4f}%")
print(f"Volatility: {ret_stats['std']*100:.4f}%")
print(f"Positive Returns: {ret_stats['positive_returns_pct']:.2f}%")

print("\n" + "-"*70)
print("CANDLE SUMMARY")
print("-"*70)
candle_stats = summary['candle_statistics']
print(f"Avg Body: ${candle_stats['avg_body']:.2f}")
print(f"Avg Range: ${candle_stats['avg_range']:.2f}")
print(f"Bullish %: {candle_stats['bullish_pct']:.2f}%")

print("\n" + "="*70)
print("✓ Exploratory analysis complete")
print("="*70)